# 11 · Season model

The year-round model (notebook 09) learned each district's crop shares over the whole year, kharif and rabi pooled,
and never saw the sowing date: Jangoan got rice and cotton for a January, March, June or October sowing alike
(user test, Nov 4). Cotton is a June crop there; rabi is rice and maize.

DES reports area by season, so the target can be per season:

- **One row per district × season × crop**, weighted by the crop's share of that district's season area
  (`data.district_season_area()`: Kharif, Autumn and aman ("Winter") rice → kharif; Rabi, other Winter crops and boro
  ("Summer") rice → rabi; other Summer crops → summer; year-round crops (sugarcane, coconut, banana, fruit) split
  equally across the three seasons, whatever DES season they are filed under: Uttar Pradesh files sugarcane as Kharif).
  District-seasons with less than 1,000 ha are left out (nothing much is sown then).
- **Inputs:** the app's 11 inputs (card N, P, K, pH; year, kharif and rabi weather) + the season (0 kharif, 1 rabi,
  2 summer). The app picks the season from the sowing month (`recommend.SEASON_MONTHS`: Jun–Sep, Oct–Jan, Feb–May).
- **Validation:** leave one state out, as in notebook 09. Scored per district-season: main crop of that season first,
  in the list (top 3 with probability ≥ 5%), share of the list grown that season, area covered.
- **Season filter:** a crop is offered in a season only if at least 5% of that season's training district-seasons grow
  it (so no wheat in summer, no chickpea in kharif). Suitability values stay the forest's own, so SHAP still adds up.
- **Compared with:** the year-round model (notebook 09's app model, refit per fold, same answer for every season) and
  per-season popularity (the 3 crops with the highest mean share in that season in the training states).
- Same 539 districts and 27 crops as notebook 09. Seed 42.

In [1]:
import sys
sys.path.insert(0, '../project')
import datetime
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import LeaveOneGroupOut

import data
import recommend

SEED = 42
MIN_DISTRICTS = 20  # as notebook 09
MIN_SEASON_HA = 1000
MIN_GROWN = 0.05  # season filter
RESULTS = Path('../research/results')
APP_FEATURES = ['N', 'P', 'K', 'ph', 'temperature', 'humidity', 'rainfall', 'rain_kharif', 'rain_rabi', 'temp_kharif', 'temp_rabi']
pd.set_option('display.width', 220)


def forest():
    return RandomForestClassifier(n_estimators=500, min_samples_leaf=3, random_state=SEED, n_jobs=-1)

## 1. Training table

In [2]:
soil = data.district_soil().dropna()
soil = soil[soil.samples >= 100]
year_area = data.district_yield()[['crop', 'district_code', 'area_ha']]
classes = sorted(year_area.crop.value_counts().loc[lambda s: s >= MIN_DISTRICTS].index)  # notebook 09's 27 crops
year_area = year_area[year_area.crop.isin(classes)]
d = recommend._tables()[2].merge(soil, on='district_code')
d = d[d.district_code.isin(year_area.district_code)].reset_index(drop=True)
d = pd.concat([d, pd.DataFrame([recommend.weather(r.lat, r.lon) for r in d.itertuples()])], axis=1)
Y = year_area.pivot_table(index='district_code', columns='crop', values='area_ha', fill_value=0).reindex(columns=classes, fill_value=0)
Y = Y.loc[d.district_code].div(Y.loc[d.district_code].sum(axis=1), axis=0).values  # year-round shares, notebook 09's target

sa = data.district_season_area()
sa = sa[sa.crop.isin(classes) & sa.district_code.isin(d.district_code)]
wide = sa.pivot_table(index=['district_code', 'season'], columns='crop', values='area_ha', fill_value=0).reindex(columns=classes, fill_value=0)
wide = wide[wide.sum(axis=1) >= MIN_SEASON_HA]
u = wide.index.to_frame(index=False).merge(d.reset_index().rename(columns={'index': 'district_row'}), on='district_code')
u['season_code'] = u.season.map(data.SEASONS.index)
S = wide.div(wide.sum(axis=1), axis=0).values  # season shares, one row per district-season
main_idx = S.argmax(axis=1)
states = u.state_name.values
print(f'{len(d)} districts, {len(u)} district-seasons, {len(classes)} crops')
pd.crosstab(np.array(classes)[main_idx], u.season).loc[lambda t: t.sum(axis=1) >= 3].sort_values('kharif', ascending=False)

539 districts, 1399 district-seasons, 27 crops


season,kharif,rabi,summer
row_0,,,
rice,353,107,0
cotton,45,1,0
maize,35,13,39
soybean,31,0,0
groundnut,16,4,16
bajra,14,0,11
ragi,8,4,0
coconut,8,22,49
blackgram,7,14,8


## 2. Leave-state-out comparison

In [3]:
FEATS = APP_FEATURES + ['season_code']


def full_proba(model, X):
    P = np.zeros((len(X), len(classes)))
    P[:, model.classes_] = model.predict_proba(X)  # a fold may miss a rare crop
    return P


def season_crops(idx):  # per season, 0/1 mask of crops grown in >= MIN_GROWN of the district-seasons idx
    return {s: ((S[idx][u.season.values[idx] == s] > 0).mean(axis=0) >= MIN_GROWN).astype(float) for s in data.SEASONS}


def long_rows(X, shares):
    i, c = np.nonzero(shares > 0)
    return X.iloc[i].reset_index(drop=True), c, shares[i, c]


P_season, P_filter, P_year, P_pop = np.zeros_like(S), np.zeros_like(S), np.zeros_like(S), np.zeros_like(S)
for tr, te in LeaveOneGroupOut().split(u, groups=states):
    Xs, ys, ws = long_rows(u.loc[tr, FEATS], S[tr])
    P_season[te] = full_proba(forest().fit(Xs, ys, sample_weight=ws), u.loc[te, FEATS])
    mask = season_crops(tr)
    P_filter[te] = P_season[te] * np.stack([mask[s] for s in u.season.values[te]])
    train_d = np.unique(u.district_row.values[tr])  # the year-round model learns from the same districts
    Xd, yd, wd = long_rows(d.loc[train_d, APP_FEATURES], Y[train_d])
    P_year[te] = full_proba(forest().fit(Xd, yd, sample_weight=wd), u.loc[te, APP_FEATURES])
    for s in data.SEASONS:
        tr_s, te_s = tr[u.season.values[tr] == s], te[u.season.values[te] == s]
        if len(te_s):
            P_pop[te_s] = S[tr_s].mean(axis=0)


def hits(P):
    order = np.argsort(-P, axis=1)[:, :3]
    keep = np.take_along_axis(P, order, axis=1) >= recommend.MIN_SUITABILITY
    keep[:, 0] = True
    picked = [o[k] for o, k in zip(order, keep)]
    return pd.DataFrame({'first': order[:, 0] == main_idx,
                         'in list': [m in p for m, p in zip(main_idx, picked)],
                         'grown': [np.mean(r[p] > 0) for r, p in zip(S, picked)],
                         'area': [r[p].sum() for r, p in zip(S, picked)],
                         'season': u.season.values, 'district': u.district_code.values})


H = {'season model + season filter (app)': hits(P_filter), 'season model': hits(P_season), 'year-round model (notebook 09)': hits(P_year), 'popularity per season': hits(P_pop)}
cols = {'first': 'main crop 1st %', 'in list': 'main crop in list %', 'grown': 'picks grown %', 'area': 'area covered %'}
overall = pd.DataFrame({k: h[list(cols)].mean() * 100 for k, h in H.items()}).T.rename(columns=cols)
overall.round(1)

,main crop 1st %,main crop in list %,picks grown %,area covered %
season model + season filter (app),52.1,76.5,87.7,68.4
season model,52.1,76.5,87.3,68.4
year-round model (notebook 09),35.0,58.0,59.1,49.5
popularity per season,47.4,67.1,79.2,59.3


In [4]:
by_season = pd.concat({k: h.groupby('season')[['first', 'in list']].mean() * 100 for k, h in H.items()}, axis=1)
by_season.columns = [f'{m} · {cols[c]}' for m, c in by_season.columns]
print('district-seasons:', u.season.value_counts().to_dict())
by_season.round(1).T

district-seasons: {'kharif': 537, 'rabi': 522, 'summer': 340}


season,kharif,rabi,summer
season model + season filter (app) · main crop 1st %,67.0,49.2,32.9
season model + season filter (app) · main crop in list %,83.8,75.1,67.1
season model · main crop 1st %,67.0,49.2,32.9
season model · main crop in list %,83.8,75.1,67.1
year-round model (notebook 09) · main crop 1st %,53.3,38.7,0.3
year-round model (notebook 09) · main crop in list %,79.0,63.4,16.5
popularity per season · main crop 1st %,65.7,40.6,28.8
popularity per season · main crop in list %,75.6,67.0,53.8


In [5]:
# Paired bootstrap, resampling districts (a district's seasons stay together)
rng = np.random.default_rng(SEED)
dist = u.district_code.values
ids = np.unique(dist)
groups = [np.flatnonzero(dist == i) for i in ids]


def boot_diff(a, b, col):
    diff = (H[a][col].astype(float) - H[b][col].astype(float)).values
    means = []
    for _ in range(2000):
        rows = np.concatenate([groups[i] for i in rng.integers(0, len(ids), len(ids))])
        means.append(diff[rows].mean())
    return {'comparison': f'{a} vs {b}', 'metric': cols[col], 'diff (pts)': diff.mean() * 100,
            '95% CI low': np.percentile(means, 2.5) * 100, '95% CI high': np.percentile(means, 97.5) * 100}


APP = 'season model + season filter (app)'
boot = pd.DataFrame([boot_diff(a, b, c) for a, b in [(APP, 'season model'), (APP, 'year-round model (notebook 09)'),
                                                     (APP, 'popularity per season')]
                     for c in ['first', 'in list']])
out = overall.copy()
out.insert(0, 'date', '2026-11-04')
out.round(2).rename_axis('method').to_csv(RESULTS / '2026-11-04_season_model_cv.csv')
by_season.round(2).to_csv(RESULTS / '2026-11-04_season_model_by_season.csv')
boot.round(1)

,comparison,metric,diff (pts),95% CI low,95% CI high
0,season model + season filter (app) vs season m...,main crop 1st %,0.0,0.0,0.0
1,season model + season filter (app) vs season m...,main crop in list %,0.0,0.0,0.0
2,season model + season filter (app) vs year-rou...,main crop 1st %,17.2,15.0,19.4
3,season model + season filter (app) vs year-rou...,main crop in list %,18.5,16.2,20.8
4,season model + season filter (app) vs populari...,main crop 1st %,4.7,2.3,7.0
5,season model + season filter (app) vs populari...,main crop in list %,9.4,7.0,11.7


## 3. App model

In [6]:
X, y, w = long_rows(u[FEATS], S)
model = forest().fit(X, y, sample_weight=w)
mask = season_crops(np.arange(len(u)))
joblib.dump({'model': model, 'forest': model, 'features': FEATS, 'classes': [classes[i] for i in model.classes_],
             'season_crops': {s: [c for c, ok in zip(classes, m) if ok] for s, m in mask.items()}},
            Path('../models/crop_model.joblib'))
print({s: len(c) for s, c in joblib.load('../models/crop_model.joblib')['season_crops'].items()}, 'crops allowed per season')
print('saved ../models/crop_model.joblib:', len(model.classes_), 'crops,', len(u), 'district-seasons')
pd.Series(model.feature_importances_, index=FEATS).sort_values(ascending=False).round(3)

{'kharif': 24, 'rabi': 23, 'summer': 17} crops allowed per season
saved ../models/crop_model.joblib: 27 crops, 1399 district-seasons


season_code    0.341
humidity       0.093
temp_rabi      0.089
rain_rabi      0.075
temp_kharif    0.065
rainfall       0.058
temperature    0.054
rain_kharif    0.053
K              0.045
ph             0.043
N              0.043
P              0.041
dtype: float64

In [7]:
# Through the app: district SHC soil (or the nearest-district fill), sown 1 Jul, 1 Nov, 1 Mar
recommend.MODEL_FILE = 'crop_model.joblib'
recommend._model.cache_clear(); recommend._explainer.cache_clear()
truth = data.district_season_area()  # all districts, incl. those without SHC data
ex = []
for name, state in [('Jangoan', 'Telangana'), ('Vidisha', 'Madhya Pradesh'), ('Ludhiana', 'Punjab'), ('Thanjavur', 'Tamil Nadu')]:
    soil_v, _ = recommend.area_soil(name, state)
    for sow in (datetime.date(2026, 7, 1), datetime.date(2026, 11, 1), datetime.date(2026, 3, 1)):
        inputs, t = recommend.recommend(*soil_v.values(), name, state, sow=sow)
        code_ = recommend.find_district(name, state).district_code
        actual = truth[(truth.district_code == code_) & (truth.season == inputs['season'])].nlargest(3, 'area_ha')
        ex.append({'district': name, 'season': inputs['season'], 'sown in season %': round(inputs['season_share'] * 100),
                   'app top 3': ', '.join(f'{c} {p:.2f}' for c, p in zip(t.crop, t.suitability)),
                   'DES top 3 that season': ', '.join(actual.crop)})
ex = pd.DataFrame(ex)
ex.to_csv(RESULTS / '2026-11-04_season_model_examples.csv', index=False)
ex

,district,season,sown in season %,app top 3,DES top 3 that season
0,Jangoan,kharif,100,"rice 0.46, cotton 0.46","cotton, rice, pigeonpeas"
1,Jangoan,rabi,44,"rice 0.88, maize 0.07","rice, maize, groundnut"
2,Jangoan,summer,0,"sugarcane 0.35, groundnut 0.11","sugarcane, banana"
3,Vidisha,kharif,99,"soybean 0.67, blackgram 0.17, rice 0.09","soybean, blackgram, rice"
4,Vidisha,rabi,100,"wheat 0.61, chickpea 0.21, lentil 0.08","wheat, chickpea, lentil"
5,Vidisha,summer,1,"mungbean 0.79, maize 0.08, blackgram 0.06","mungbean, maize, blackgram"
6,Ludhiana,kharif,100,"rice 0.56, sugarcane 0.19, maize 0.06","rice, maize, sugarcane"
7,Ludhiana,rabi,97,"wheat 0.73, sugarcane 0.12, rice 0.05","wheat, mungbean, mustard"
8,Ludhiana,summer,0,"sugarcane 0.57, wheat 0.20, mungbean 0.08",sugarcane
9,Thanjavur,kharif,100,"rice 0.85, coconut 0.05","rice, coconut, blackgram"


## Findings

Run of Nov 4 after the web check: year-round crops (sugarcane, coconut, banana, fruit) are split equally across the
three seasons whatever DES calls them. The first run counted their "Whole Year" area in full in every season (and UP's
sugarcane, filed as Kharif, only in kharif), so sugarcane and coconut topped seasons they barely fill.

1. **Season matters more than anything else the model sees** (importance 0.34; next humidity 0.09).
2. **The app model gets the season's main crop into the list for 76.5% of district-seasons and first for 52.1%**;
   the year-round model 58.0% / 35.0% (+18.5 pts in list, CI 16.2–20.8) and per-season popularity 67.1% / 47.4%
   (+9.4, CI 7.0–11.7; first +4.7, CI 2.3–7.0).
3. **By season (in list):** kharif 83.8%, rabi 75.1%, summer 67.1%. The year-round model is near useless in summer
   (16.5%).
4. **The season filter changes nothing measurable**; kept because it stops crops nobody sows in that season.
5. Still open: summer is often nearly empty (the app warns below 10% of the main season's area); districts without
   SHC data take soil and, in effect, cropping from neighbours (Karnal gets UP's sugarcane); yields and prices are
   year-round. See `research/paper_limitations.md`.